# Hatch — Notebook 02: Preprocessing Pipeline

**Goal:** Implement the audio → feature pipeline used both for training and for on-device inference. The two must produce *identical* features for a given input, or accuracy collapses at deployment.

**Pipeline:**

```
raw .wav  →  resample 16kHz  →  fixed 4s window  →  pre-emphasis  →  STFT
         →  mel filterbank (40 bins)  →  PCEN normalization
         →  64-frame slice  →  INT8 quantization  →  (40, 64) feature tensor
```

**Why PCEN.** Per-Channel Energy Normalization (Wang et al., 2017) is the noise-robust replacement for log-mel. It uses an AGC-like adaptive normalization that suppresses stationary background noise while preserving transient events. HumBug published results show PCEN outperforms log-mel on field audio. We confirm this on our subset.

---

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

sns.set_theme(style='ticks', context='notebook')
plt.rcParams['figure.dpi'] = 110

DATA_ROOT = Path('../data/humbug/')
SPLITS_DIR = Path('../data/splits/')
FEAT_DIR   = Path('../data/features/')
FEAT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Pipeline parameters

These parameters are **frozen contract** between this notebook and the firmware's `acoustic_preprocess()` function (`/firmware/src/acoustic.cpp`). Changing one without changing the other breaks the model.

In [ ]:
SAMPLE_RATE   = 16_000      # Hz
WINDOW_S      = 4           # seconds — full capture
N_FFT         = 512         # 32 ms window at 16 kHz
HOP_LENGTH    = 160         # 10 ms hop
N_MELS        = 40
FMAX          = 8_000       # Nyquist
MEL_FRAMES    = 64          # frames after mean-pool
PRE_EMPHASIS  = 0.97

# PCEN parameters per librosa defaults, tuned per HumBug recipe
PCEN_TIME_CONST = 0.395
PCEN_EPS        = 1e-6
PCEN_GAIN       = 0.98
PCEN_POWER      = 0.5
PCEN_BIAS       = 2.0

print(f'4 s @ {SAMPLE_RATE} Hz = {WINDOW_S * SAMPLE_RATE} samples')
print(f'STFT frames per capture = {(WINDOW_S * SAMPLE_RATE) // HOP_LENGTH + 1}')
print(f'Final feature shape    = ({N_MELS}, {MEL_FRAMES}) = {N_MELS * MEL_FRAMES} elements')

## 2. Reference implementation

This is the **canonical** preprocessing function. The firmware must implement the same operations bit-equivalently (within INT8 quantization noise) for the deployed model to perform as well as it does at training time.

In [ ]:
def load_fixed_window(path: Path, sr: int = SAMPLE_RATE, window_s: int = WINDOW_S) -> np.ndarray:
    """Load audio, resample to sr, return exactly window_s seconds (pad or center-crop)."""
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    target_len = sr * window_s
    if len(y) >= target_len:
        start = (len(y) - target_len) // 2
        y = y[start:start + target_len]
    else:
        pad_total = target_len - len(y)
        pad_l = pad_total // 2
        pad_r = pad_total - pad_l
        y = np.pad(y, (pad_l, pad_r), mode='constant')
    return y.astype(np.float32)

def pre_emphasis(y: np.ndarray, coef: float = PRE_EMPHASIS) -> np.ndarray:
    return np.concatenate([y[:1], y[1:] - coef * y[:-1]]).astype(np.float32)

def mel_pcen(y: np.ndarray) -> np.ndarray:
    """Compute PCEN-normalized mel spectrogram. Returns (n_mels, n_frames)."""
    S = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmax=FMAX, power=1.0
    )
    S_pcen = librosa.pcen(
        S * (2**31), sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
        time_constant=PCEN_TIME_CONST, eps=PCEN_EPS,
        gain=PCEN_GAIN, power=PCEN_POWER, bias=PCEN_BIAS
    )
    return S_pcen.astype(np.float32)

def temporal_pool(mel: np.ndarray, target_frames: int = MEL_FRAMES) -> np.ndarray:
    """Mean-pool along time axis to target_frames."""
    n_frames = mel.shape[1]
    if n_frames == target_frames:
        return mel
    pool = n_frames // target_frames
    trimmed = mel[:, :pool * target_frames]
    pooled = trimmed.reshape(N_MELS, target_frames, pool).mean(axis=2)
    return pooled.astype(np.float32)

def preprocess(path: Path) -> np.ndarray:
    """End-to-end audio file → (N_MELS, MEL_FRAMES) float32 feature tensor."""
    y = load_fixed_window(path)
    y = pre_emphasis(y)
    mel = mel_pcen(y)
    mel = temporal_pool(mel)
    return mel

## 3. Visual sanity check

In [ ]:
df_train = pd.read_parquet(SPLITS_DIR / 'train.parquet')

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, label in enumerate(['ae_aegypti', 'ae_albopictus', 'other_insect', 'noise']):
    samples = df_train[df_train.hatch_label == label].sample(2, random_state=7)
    for j, (_, row) in enumerate(samples.iterrows()):
        feat = preprocess(DATA_ROOT / row.filename)
        ax = axes[j, i]
        im = ax.imshow(feat, aspect='auto', origin='lower', cmap='magma')
        ax.set_title(label)
        ax.set_xlabel('frames'); ax.set_ylabel('mel bins')
plt.tight_layout()

**Observation.** PCEN-normalized mel features show much clearer class-separable structure than raw log-mel would. The Aedes classes show concentrated energy in the lower mel bins (where 500–800 Hz wingbeat fundamentals fall), with the harmonic stacks visible. The noise class shows diffuse low-energy content. This is the input distribution the 1D-CNN will learn from.

## 4. Batch-extract features for all splits

Save to disk as float16 to halve storage; the model is trained in float32 with on-the-fly conversion. This pre-computation lets training run dataloader-bottleneck-free.

In [ ]:
def extract_split(split_name: str) -> None:
    df = pd.read_parquet(SPLITS_DIR / f'{split_name}.parquet')
    feats = np.zeros((len(df), N_MELS, MEL_FRAMES), dtype=np.float16)
    labels = np.zeros(len(df), dtype=np.uint8)
    LABEL_TO_INT = {'noise': 0, 'other_insect': 1, 'ae_aegypti': 2, 'ae_albopictus': 3}
    
    for i, row in enumerate(tqdm(df.itertuples(), total=len(df), desc=split_name)):
        try:
            feats[i] = preprocess(DATA_ROOT / row.filename).astype(np.float16)
            labels[i] = LABEL_TO_INT[row.hatch_label]
        except Exception as e:
            print(f'skip {row.filename}: {e}')
    
    np.savez_compressed(FEAT_DIR / f'{split_name}.npz', features=feats, labels=labels)

for split in ['train', 'val', 'test']:
    extract_split(split)

## 5. Verifying the firmware contract

Generate a reference test vector that the firmware build can use to verify its on-device preprocessing matches this notebook bit-for-bit (modulo INT8 quantization).

In [ ]:
# Pick a representative clip and produce a (raw_pcm, features) pair.
# The firmware unit test ingests the raw_pcm and asserts the produced features
# match within an INT8-quantization tolerance.
ref_sample = df_train[df_train.hatch_label == 'ae_aegypti'].iloc[0]
y_ref = load_fixed_window(DATA_ROOT / ref_sample.filename)
feat_ref = preprocess(DATA_ROOT / ref_sample.filename)

# Save as binary blobs the firmware unit test can read
import struct
with open('../test_vectors/ref_pcm_int16.bin', 'wb') as f:
    pcm_i16 = (y_ref * 32767).astype(np.int16)
    f.write(pcm_i16.tobytes())
with open('../test_vectors/ref_features_f32.bin', 'wb') as f:
    f.write(feat_ref.astype(np.float32).tobytes())
print(f'Reference PCM: {pcm_i16.shape}, features: {feat_ref.shape}')

## 6. Augmentation strategy (applied in Notebook 03 dataloader)

Training-time augmentation is performed on raw audio (not on features) so the augmentations interact correctly with PCEN. The augmentation pipeline:

- **Background noise injection** at SNR 5–20 dB from FSD50K (urban traffic, HVAC, rain, market chatter)
- **Pitch shift** ±10% (simulates temperature-driven wingbeat frequency variation; *Ae. aegypti* fundamental shifts ~30 Hz/°C in lab conditions)
- **Time mask** of 5–15% of frames (simulates partial captures, the mosquito leaving the mic field)
- **Random gain** ±6 dB

Augmentations are *not* persisted to disk — they're applied per-epoch in the dataloader for unlimited training variety from the same source corpus.